# EquiFrame Instalación y primera comparación con datos reales

Este cuaderno explica cómo instalar una versión identificada de EquiFrame,
conservar **10.000 registros reales**, comprobar tres formatos y aceptar o
rechazar una comparación. En Colab, selecciona **Entorno de ejecución →
Ejecutar todas** con CPU y Python compatible (3.10 a 3.13). No se requiere GPU.
Las descargas necesitan conexión a Internet. Ejecuta las celdas en orden.


In [1]:
import subprocess
import sys

# Archivo público inmutable de la versión auditada. No requiere fichas API.
wheel = (
    "https://github.com/htroya/equiframe/releases/download/v0.14.0/"
    "equiframe-0.14.0-py3-none-any.whl"
    "#sha256=c1956d8cfeca3b558fccda9955d1cdba235934041df67ed753cca698ed461219"
)
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet",
    "equiframe[engines] @ " + wheel,
    "pandas==3.1.0", "polars==1.44.2", "duckdb==1.5.6", "pyarrow==25.0.1",
])
subprocess.check_call([sys.executable, "-m", "pip", "check"])
print("Instalación de EquiFrame 0.14.0 comprobada.")


Instalación de EquiFrame 0.14.0 comprobada.


## Fuente y selección de los datos

Se descargan datos reales de **E-Commerce Data**, versión 1, desde
[Kaggle](https://www.kaggle.com/datasets/carrie1/ecommerce-data).
Su fuente original es **Online Retail**, de Daqing Chen, publicada por UCI:
[https://doi.org/10.24432/C5BW33](https://doi.org/10.24432/C5BW33).
UCI declara la licencia [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).
Esta atribución se conserva; no se atribuye a Kaggle una licencia CC0.

Se seleccionan las **primeras 10.000 líneas de transacción**, sin replicar registros,
eliminar devoluciones ni suprimir clientes ausentes. Es una selección didáctica
determinista, no una muestra representativa para inferencia estadística.
La descarga pública no usa cuentas, contraseñas ni claves API.

La clave técnica `row_id` identifica la posición de cada registro seleccionado;
no sustituye un identificador de negocio. Los identificadores originales se
conservan como texto. Los campos vacíos de descripción y cliente se interpretan
como ausencias conforme a una convención explícita de lectura. La fecha se
interpreta como mes/día/año, sin inventar una zona horaria. Los precios se
conservan como `Decimal`; `amount_minor` expresa el importe firmado en peniques
de GBP y mantiene las cantidades negativas de los registros originales.


In [2]:
import csv
import hashlib
import io
import itertools
import json
import tempfile
import zipfile
from datetime import datetime
from decimal import Decimal
from pathlib import Path
from urllib.request import Request, urlopen

import equiframe as ef
import pandas as pd
import polars as pl
import pyarrow as pa
import pyarrow.parquet as pq

N = 10_000
source_url = (
    "https://www.kaggle.com/api/v1/datasets/download/carrie1/ecommerce-data"
    "?datasetVersionNumber=1"
)
with urlopen(Request(source_url, headers={"User-Agent": "EquiFrame-public-tutorial/1.0"}),
             timeout=60) as response:
    payload = response.read(60_000_001)
if len(payload) > 60_000_000:
    raise RuntimeError("La descarga excede el límite declarado.")
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    if archive.namelist() != ["data.csv"]:
        raise RuntimeError("El inventario de la fuente cambió.")
    csv_bytes = archive.read("data.csv")
source_sha256 = hashlib.sha256(csv_bytes).hexdigest()
if source_sha256 != "d07aec9960083af2339975a3f9d3b26313b342dcd9f86cce0b919b1cde639a44":
    raise RuntimeError("La fuente no coincide con la versión examinada.")
reader = csv.DictReader(io.StringIO(csv_bytes.decode("ISO-8859-1"), newline=""))
original_rows = list(itertools.islice(reader, N))
source_rows = N + sum(1 for _ in reader)
if len(original_rows) != N or source_rows != 541_909:
    raise RuntimeError("El tamaño de la fuente o de la selección cambió.")

records = []
for row_id, row in enumerate(original_rows, 1):
    price = Decimal(row["UnitPrice"])
    quantity = int(row["Quantity"])
    minor = price * quantity * 100
    if price.as_tuple().exponent < -2 or minor != minor.to_integral_value():
        raise RuntimeError("El precio requiere otra escala monetaria.")
    records.append({
        "row_id": row_id, "invoice_no": row["InvoiceNo"],
        "stock_code": row["StockCode"], "description": row["Description"] or None,
        "quantity": quantity,
        "invoice_date": datetime.strptime(row["InvoiceDate"], "%m/%d/%Y %H:%M"),
        "unit_price": price, "customer_id": row["CustomerID"] or None,
        "country": row["Country"], "currency": "GBP", "amount_minor": int(minor),
    })
schema = pa.schema([
    ("row_id", pa.int64()), ("invoice_no", pa.string()), ("stock_code", pa.string()),
    ("description", pa.string()), ("quantity", pa.int64()),
    ("invoice_date", pa.timestamp("us")), ("unit_price", pa.decimal128(18, 2)),
    ("customer_id", pa.string()), ("country", pa.string()),
    ("currency", pa.string()), ("amount_minor", pa.int64()),
])
original_table = pa.Table.from_pylist(records, schema=schema)
temporary = tempfile.TemporaryDirectory(prefix="equiframe-kaggle-")
work = Path(temporary.name)
print(json.dumps({
    "version_equiframe": ef.__version__, "filas_fuente": source_rows,
    "filas_seleccionadas": len(records), "sha256_csv_original": source_sha256,
    "clientes_ausentes": sum(r["customer_id"] is None for r in records),
    "descripciones_ausentes": sum(r["description"] is None for r in records),
    "cantidades_negativas": sum(r["quantity"] < 0 for r in records),
}, ensure_ascii=False, indent=2))


{
  "version_equiframe": "0.14.0",
  "filas_fuente": 541909,
  "filas_seleccionadas": 10000,
  "sha256_csv_original": "d07aec9960083af2339975a3f9d3b26313b342dcd9f86cce0b919b1cde639a44",
  "clientes_ausentes": 2291,
  "descripciones_ausentes": 42,
  "cantidades_negativas": 130
}


## Conservar los mismos registros en CSV JSONL y Parquet

Declarar los tipos evita que los identificadores se conviertan en números o
que los precios pasen por punto flotante. La comprobación siguiente compara
los 10.000 registros completos, incluidas sus repeticiones y ausencias.


In [3]:
# CSV y JSONL utilizan representaciones textuales explícitas para Decimal y fechas.
# Parquet conserva sus tipos. La misma especificación reconstruye las tres entradas.
serial = [dict(r, invoice_date=r["invoice_date"].isoformat(),
               unit_price=format(r["unit_price"], ".2f")) for r in records]
csv_path, jsonl_path, parquet_path = work / "retail.csv", work / "retail.jsonl", work / "retail.parquet"
with csv_path.open("w", encoding="utf-8", newline="") as stream:
    writer = csv.DictWriter(stream, fieldnames=schema.names)
    writer.writeheader()
    writer.writerows(serial)
with jsonl_path.open("w", encoding="utf-8") as stream:
    for row in serial:
        stream.write(json.dumps(row, ensure_ascii=False) + "\n")
pq.write_table(original_table, parquet_path)

def typed(rows):
    restored = []
    for item in rows:
        row = dict(item)
        for name in ("row_id", "quantity", "amount_minor"):
            row[name] = int(row[name])
        row["invoice_date"] = datetime.fromisoformat(row["invoice_date"])
        row["unit_price"] = Decimal(row["unit_price"])
        for name in ("description", "customer_id"):
            if row[name] == "":
                row[name] = None
        restored.append(row)
    return pa.Table.from_pylist(restored, schema=schema)

with csv_path.open(encoding="utf-8", newline="") as stream:
    csv_table = typed(csv.DictReader(stream))
with jsonl_path.open(encoding="utf-8") as stream:
    jsonl_table = typed(json.loads(line) for line in stream)
parquet_table = pq.read_table(parquet_path)
sources = {"CSV": csv_table, "JSONL": jsonl_table, "Parquet": parquet_table}
policy = ef.ComparisonPolicy(
    index="ignore", physical_types="lossless",
    justification="Las columnas declaradas conservan el significado; el índice de Pandas no es una clave.",
)
for name, table in sources.items():
    result = ef.diff(table, original_table, keys=["row_id"], policy=policy,
                     work_dir=work / ("format-" + name))
    ef.assert_equivalent(result)
    print(name, table.num_rows, result.state, "completa:", result.complete)


CSV 10000 equivalent completa: True
JSONL 10000 equivalent completa: True
Parquet 10000 equivalent completa: True


## Primera comparación y control de una diferencia

Se contrasta una tabla de Pandas con la captura tipada del Parquet. Después
se modifica deliberadamente una cantidad para comprobar el rechazo.
Esta modificación es un control didáctico; no se atribuye a los datos originales.


In [4]:
frame = csv_table.to_pandas(types_mapper=pd.ArrowDtype)
same = ef.diff(frame, parquet_table, keys=["row_id"], policy=policy,
               work_dir=work / "basic-equivalent")
ef.assert_equivalent(same)
print("Comparación original:", same.state, "completa:", same.complete)

changed = frame.copy(deep=True)
changed.loc[0, "quantity"] = int(changed.loc[0, "quantity"]) + 1
different = ef.diff(changed, parquet_table, keys=["row_id"], policy=policy,
                    work_dir=work / "basic-different")
if different.state != "different" or not different.complete:
    raise RuntimeError("No se obtuvo la divergencia esperada del control.")
try:
    ef.assert_equivalent(different)
except ef.EquivalenceDifferentError:
    print("Control modificado:", different.state, "aceptación rechazada.")
else:
    raise RuntimeError("El control modificado no debe aceptarse.")
summary = {"records": N, "formats_verified": 3,
           "equivalent": same.state, "negative_case": different.state}
print(json.dumps(summary, ensure_ascii=False))


Comparación original: equivalent completa: True
Control modificado: different aceptación rechazada.
{"records": 10000, "formats_verified": 3, "equivalent": "equivalent", "negative_case": "different"}


## Interpretación y conservación

`equivalent` junto con una evaluación completa acredita el cumplimiento del
contrato sobre las entradas examinadas. `different` conserva una divergencia;
`not_evaluable` y una evaluación incompleta no autorizan aceptar los datos.
`assert_equivalent` aplica esta condición de aceptación sin sustituirla por
la ausencia de mensajes de error.

Los resultados de este cuaderno documentan el uso de la biblioteca. No son una
estimación de la eficacia de Isolation Forest ni resultados del corpus SERCOP.
Los datos se descargan al ejecutar y no se incluyen en el repositorio del software.
Las salidas publicadas muestran únicamente resúmenes técnicos, sin registros de clientes.

**Referencias**

Chen, D. (2015). *Online Retail* [Conjunto de datos]. UCI Machine Learning Repository.
https://doi.org/10.24432/C5BW33

Troya, H. (2026). *EquiFrame* (Versión 0.14.0) [Software]. GitHub.
https://github.com/htroya/equiframe/releases/tag/v0.14.0


In [5]:
# Elimina exclusivamente los archivos temporales creados por este cuaderno.
temporary.cleanup()
print("Datos y capturas temporales eliminados.")


Datos y capturas temporales eliminados.
